In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("../data/raw/Student_Performance.csv")

In [3]:
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [4]:
df["Extracurricular Activities"].unique()

array(['Yes', 'No'], dtype=object)

In [5]:
df["Extracurricular Activities"]=df["Extracurricular Activities"].map({"Yes":1,"No":0})

In [6]:
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0


In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X=df.drop("Performance Index",axis=1)
y=df["Performance Index"]

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
from sklearn.linear_model import LinearRegression

In [11]:
model=LinearRegression()

In [12]:
model.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [13]:
#Model parameters
print("Coefficients",model.coef_)
print("Intercept: ",model.intercept_)

Coefficients [2.85248393 1.0169882  0.60861668 0.47694148 0.19183144]
Intercept:  -33.92194621555631


In [14]:
predictions=model.predict(X_test)

In [15]:
predictions[:10]

array([54.71185392, 22.61551294, 47.90314471, 31.28976748, 43.00457042,
       59.07125171, 45.90347516, 86.45911791, 37.70014037, 72.05592481])

In [16]:
comparisons=pd.DataFrame({
    "Actual":y_test,
    "Predicted":predictions
})
comparisons.head(10)

,Actual,Predicted
6252,51.0,54.711854
4684,20.0,22.615513
1731,46.0,47.903145
4742,28.0,31.289767
4521,41.0,43.004570
6340,59.0,59.071252
576,48.0,45.903475
5202,87.0,86.459118
6363,37.0,37.700140
439,73.0,72.055925


In [17]:
comparisons["Error"]=comparisons["Actual"]-comparisons["Predicted"]
comparisons.head()

,Actual,Predicted,Error
6252,51.0,54.711854,-3.711854
4684,20.0,22.615513,-2.615513
1731,46.0,47.903145,-1.903145
4742,28.0,31.289767,-3.289767
4521,41.0,43.004570,-2.004570


In [18]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [19]:
mae=mean_absolute_error(y_test,predictions)
mse=mean_squared_error(y_test,predictions)
r2=r2_score(y_test,predictions)
rmse=np.sqrt(mse)
print(f"MAE:{mae}")
print(f"MSE:{mse}")
print(f"RMSE:{rmse}")
print(f"R2 Score:{r2}")

MAE:1.6111213463123044
MSE:4.082628398521855
RMSE:2.020551508505006
R2 Score:0.9889832909573145


In [20]:
from sklearn.tree import DecisionTreeRegressor

In [24]:
tree_model=DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train,y_train)
tree_predictions=tree_model.predict(X_test)

In [29]:
tree_mae=mean_absolute_error(y_test,tree_predictions)
tree_mse=mean_squared_error(y_test,tree_predictions)
tree_rmse=np.sqrt(mse)
tree_r2=r2_score(y_test,predictions)
print(f"MAE:{tree_mae}")
print(f"MSE:{tree_mse}")
print(f"RMSE:{tree_rmse}")
print(f"R2 Score:{tree_r2}")
accuracy=pd.DataFrame(
    {
        "Linear Regression":[mae,mse,rmse,r2],
        "Decision Tree":[tree_mae,tree_mse,tree_rmse,tree_r2]
    },index=["MAE","MSE","RMSE","R2"]
)

MAE:2.3378333333333337
MSE:8.812694444444444
RMSE:2.020551508505006
R2 Score:0.9889832909573145


In [30]:
accuracy

,Linear Regression,Decision Tree
MAE,1.611121,2.337833
MSE,4.082628,8.812694
RMSE,2.020552,2.020552
R2,0.988983,0.988983


In [31]:
from sklearn.model_selection import cross_val_score

In [33]:
scores=cross_val_score(
    model,
    X,y,cv=5,scoring='r2'
)
scores

array([0.98879624, 0.98827438, 0.9891418 , 0.989087  , 0.98836991])

In [34]:
print(scores.mean())

0.9887338678935833


In [35]:
from sklearn.model_selection import GridSearchCV
params_grid={
    "max_depth":[1,2,3,4,5,6,7,8,9,10]
}

In [36]:
grid=GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid=params_grid,
    cv=5,
    scoring="r2"
)

In [39]:
grid.fit(X_train,y_train)
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 9}
0.9828232003651415
